In [1]:
import pdfplumber
import pandas as pd
import re

def extrair_pitstops_regex(caminho_pdf, caminho_csv_saida):
    print(f"Lendo o PDF como texto")
    
    dados_totais = []
    
    # O "Molde" Regex: Ele ensina o Python a reconhecer a anatomia da linha do PDF
    # ^(\d+) -> O número do carro no início
    # (.+?) -> O nome do Piloto e Equipe
    # (\d+) -> O número da Volta (Lap)
    # (\d{2}:\d{2}:\d{2}\.\d{3}) -> O Time of Day 
    # ([\d:\.]+) -> O tempo de Pit Stop no final 
    padrao = re.compile(r"^(\d+)\s+(.+?)\s+(\d+)\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s*([\d:\.]+)?$")

    # Abre o PDF
    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            for pagina in pdf.pages:
                # Em vez de procurar tabelas, extrai todo o texto da página
                texto = pagina.extract_text()
                
                if texto:
                    linhas = texto.split('\n') # Quebra o texto linha por linha
                    
                    for linha in linhas:
                        match = padrao.match(linha.strip())
                        if match:
                            # Se a linha encaixar no nosso molde, fatiamos os dados
                            carro = match.group(1)
                            piloto_equipe = match.group(2).strip()
                            volta = match.group(3)
                            time_of_day = match.group(4)
                            pit_time = match.group(5)
                            
                            # Ignora se por acaso a linha não tiver o tempo final anotado
                            if pit_time:
                                dados_totais.append([carro, piloto_equipe, volta, time_of_day, pit_time])
    except FileNotFoundError:
        print(f"ALERTA: O arquivo {caminho_pdf} não foi encontrado.")
        return

    if not dados_totais:
        print("Nenhum pit stop encontrado! O padrão do texto pode estar diferente.")
        return

    # Transforma em planilha com as colunas certinhas
    df_pitstops = pd.DataFrame(dados_totais, columns=['Carro', 'Piloto_Equipe', 'Lap', 'Time of Day', 'Pit Time'])
    
    # Exporta para CSV padrão
    df_pitstops.to_csv(caminho_csv_saida, index=False, sep=';', encoding='utf-8-sig')
    print(f"Pit Stops extraídos com perfeição! Salvo em: {caminho_csv_saida}")

# --- ÁREA DE EXECUÇÃO ---
arquivo_pdf_pit = '../data/01_raw/pit_curvelo_p1.pdf'
arquivo_csv_pit = '../data/02_interim/dados_pitstops_P1.csv'

extrair_pitstops_regex(arquivo_pdf_pit, arquivo_csv_pit)

Lendo o PDF como texto
Pit Stops extraídos com perfeição! Salvo em: ../data/02_interim/dados_pitstops_P1.csv
